# Tuning-curve comparison

Load prepared HD-class-3 and 2-D-selected RF CSVs from `data/tc_comparison/`. Each file stores complete unit identity in `unit_id` and responses under degree columns. Each load block prepares missing comparison CSVs from its explicitly named sources; existing CSVs are reused without reading the sources. RF preparation uses saved 2-D detections and creates a missing native projection, without rerunning detection.

`rf_only` chooses the saved RF-only or all-row projection. Native zero-bin selection, resampling, normalization, comparison, and plotting are explicit below; `load_tc` only reads and `concat` only stacks.


In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
from pathlib import Path
from textwrap import fill
from IPython.display import display
from matplotlib import pyplot as plt
from matplotlib.ticker import MaxNLocator
import numpy as np
from Utils.direction_comparison import load_tc, resample_profiles, rf_pick, concat, normalize_tc, prepare_comparison, plot_comparison_heatmaps, profile_coordinates, tcRange
from Utils.tc_preparation import prepare_hd_tc, prepare_rf_comparison
from Utils.rflocate import rf_result_path
from Utils.plotting import LIGHT_PLOT_STYLE, plot_direction_comparison
from Utils.statistic_utils import compare_direction_angles
from Utils.tuning_curve_utils import rayleigh_test

root = Path("/mnt/senzailab/Kai/#Recording")
rf_only: bool = True  # Choose the saved RF-only or all-row projection.
rf_suffix = "_rfonly" if rf_only else ""
rf_response_window_s = (0.0, 0.2)
rf_max_zero_bins = 2  # Skip RF units with > N zero native 1-D bins; None disables.
comparison_bins = 30
plot_options = dict(figsize=(10, 10), vmin=0, vmax=1, colorbar_label="Normalized response")
is_wrap = True


In [ ]:
hd_session_dir = root / "m14/260609/260609_1"
hd_csv_path = hd_session_dir / "data/tc_comparison/hd_class3_ProbeA.csv"
prepare_hd_tc(
    hd_session_dir / "data/tuning_curves/ProbeA/tuning_curves.tc",
    hd_csv_path, bins=comparison_bins, hd_class=3, unit_prefix="m14:260609:A",
)
fm_m14_260609 = load_tc(
    hd_csv_path,
    label="m14 260609 ProbeA HD", range=tcRange(False), response_units="Hz",
)
rf_session_dir = root / "m14/260609/260609_3"
rf_source = (rf_session_dir / "data/rfmapping/good/-100_400_1ms/ProbeA"
             / "regular_unitsSpikeCounts_260609_3.rfmap")
rf_csv_path = rf_session_dir / f"data/tc_comparison/rf_inhibitory_x_2d{rf_suffix}_ProbeA.csv"
prepare_rf_comparison(
    rf_source, rf_csv_path,
    projection_path=rf_session_dir / "data" / f"{rf_source.stem}_ProbeA{'_inhibitory' if rf_only else ''}_1d{rf_suffix}.csv",
    detected_rf_path=rf_result_path(rf_source, rf_type="inhibitory"),
    time_range_s=rf_response_window_s, rf_only=rf_only, unit_prefix="m14:260609:A",
)
rf_m14_260609 = load_tc(
    rf_csv_path,
    response_units="spike_count", label="m14 260609 ProbeA inhibitory RF3",
)
rf_m14_260609 = rf_pick(rf_m14_260609, max_zero_bins=rf_max_zero_bins)
rf_m14_260609 = resample_profiles(rf_m14_260609, range=[-150, 150], bins=comparison_bins, fill_value=0)

units_m14_260609 = fm_m14_260609.index.intersection(rf_m14_260609.index)
hd_normalized_m14_260609 = normalize_tc(fm_m14_260609.loc[units_m14_260609])
rf_normalized_m14_260609 = normalize_tc(rf_m14_260609.loc[units_m14_260609])


In [ ]:
# Native angles, one explicitly selected reference per call.
sorted_fm_m14_260609 = prepare_comparison(hd_normalized_m14_260609, rf_normalized_m14_260609, mode="native", matched_min=True)
plot_comparison_heatmaps(sorted_fm_m14_260609, **plot_options)
sorted_rf_m14_260609 = prepare_comparison(rf_normalized_m14_260609, hd_normalized_m14_260609, mode="native", reference_min=True)
plot_comparison_heatmaps(sorted_rf_m14_260609, **plot_options)


In [ ]:
# Subtract the selected reference direction from both curves.
aligned_fm_m14_260609 = prepare_comparison(hd_normalized_m14_260609, rf_normalized_m14_260609, mode="aligned", matched_min=True, matched_fill_value=0)
plot_comparison_heatmaps(aligned_fm_m14_260609, **plot_options)
aligned_rf_m14_260609 = prepare_comparison(rf_normalized_m14_260609, hd_normalized_m14_260609, mode="aligned", reference_min=True, reference_fill_value=0)
plot_comparison_heatmaps(aligned_rf_m14_260609, **plot_options)


In [ ]:
# Center the reference; add its selected direction to the matched curve.
summed_fm_m14_260609 = prepare_comparison(hd_normalized_m14_260609, rf_normalized_m14_260609, mode="sum", is_wrap=is_wrap, matched_min=True, matched_fill_value=0)
plot_comparison_heatmaps(summed_fm_m14_260609, **plot_options)
# summed_rf_m14_260609 = prepare_comparison(rf_normalized_m14_260609, hd_normalized_m14_260609, mode="sum", is_wrap=is_wrap, reference_min=True, reference_fill_value=0)
# plot_comparison_heatmaps(summed_rf_m14_260609, **plot_options)


In [ ]:
peaks = summed_fm_m14_260609["peaks"]
peak_sum_deg = (peaks["reference_peak_deg"] + peaks["matched_peak_deg"]) % 360
peak_sum_rad = np.deg2rad(peak_sum_deg.dropna().to_numpy())

counts, edges = np.histogram(peak_sum_rad, bins=30, range=(0, 2 * np.pi))
centers_deg = np.rad2deg((edges[:-1] + edges[1:]) / 2)
# Equal occupancy defines a uniform null; retain all bins, including zeros.
rayleigh_score, rayleigh_p = rayleigh_test(counts, np.ones_like(counts), centers_deg)

with plt.rc_context(LIGHT_PLOT_STYLE):
    fig, ax = plt.subplots(figsize=(6, 6), subplot_kw={"projection": "polar"}, layout="constrained")
    ax.bar((edges[:-1] + edges[1:]) / 2, counts, width=np.diff(edges), edgecolor="white")
    ax.set_theta_zero_location("N")
    ax.set_theta_direction(-1)
    ax.yaxis.set_major_locator(MaxNLocator(integer=True))
    ax.set_xlabel("Unit count (radial axis)")
    ax.set_title(
        "m14 260609: HD peak + inhibitory RF minimum angles\n"
        f"n = {int(counts.sum())}\n"
        f"Rayleigh score = {rayleigh_score:.3f}, p = {rayleigh_p:.3g}",
        pad=20,
    )
    plt.show()


In [ ]:
peaks_m14_260609 = sorted_fm_m14_260609["peaks"].dropna(subset=["reference_peak_deg", "matched_peak_deg"])
statistics_m14_260609 = None
if len(peaks_m14_260609) >= 3:
    reference = peaks_m14_260609.reference_peak_deg.to_numpy()
    matched = peaks_m14_260609.matched_peak_deg.to_numpy()
    statistics_m14_260609 = compare_direction_angles(reference, matched, n_permutations=10_000, random_seed=1)
    statistics_figures_m14_260609 = plot_direction_comparison(
        reference, matched, statistics_m14_260609,
        reference_label=fm_m14_260609.attrs["label"], matched_label=rf_m14_260609.attrs["label"],
        reference_ticklabels=profile_coordinates(fm_m14_260609)["ticklabels"],
        matched_ticklabels=profile_coordinates(rf_m14_260609)["ticklabels"],
        show=False,
    )
    for figure, axis in statistics_figures_m14_260609:
        axis.set_title(fill(axis.get_title(), width=60))
        figure.tight_layout()
        display(figure)
        plt.close(figure)


## m15 260630: HD 1 / inhibitory RF 3

Probes **A + B**, combined with probe identity retained. HD class is selected by `hd_class`.
Load the saved Probe A and Probe B curves, then combine them with `concat`.


In [ ]:
hd_session_dir = root / "m15/260630/260630_1"
hd_csv_path = hd_session_dir / "data/tc_comparison/hd_class3_ProbeA.csv"
prepare_hd_tc(
    hd_session_dir / "data/tuning_curves/ProbeA/tuning_curves.tc",
    hd_csv_path, bins=comparison_bins, hd_class=3, unit_prefix="m15:260630:A",
)
fm_m15_260630_A = load_tc(
    hd_csv_path,
    label="m15 260630 ProbeA HD", range=tcRange(False), response_units="Hz",
)
rf_session_dir = root / "m15/260630/260630_3"
rf_source = (rf_session_dir / "data/rfmapping/good/-100_400_1ms/ProbeA"
             / "regular_unitsSpikeCounts_260630_3.rfmap")
rf_csv_path = rf_session_dir / f"data/tc_comparison/rf_inhibitory_x_2d{rf_suffix}_ProbeA.csv"
prepare_rf_comparison(
    rf_source, rf_csv_path,
    projection_path=rf_session_dir / "data" / f"{rf_source.stem}_ProbeA{'_inhibitory' if rf_only else ''}_1d{rf_suffix}.csv",
    detected_rf_path=rf_result_path(rf_source, rf_type="inhibitory"),
    time_range_s=rf_response_window_s, rf_only=rf_only, unit_prefix="m15:260630:A",
)
rf_m15_260630_A = load_tc(
    rf_csv_path,
    response_units="spike_count", label="m15 260630 ProbeA inhibitory RF3",
)
rf_m15_260630_A = rf_pick(rf_m15_260630_A, max_zero_bins=rf_max_zero_bins)
rf_m15_260630_A = resample_profiles(rf_m15_260630_A, range=tcRange(True), bins=comparison_bins, fill_value=0)

hd_session_dir = root / "m15/260630/260630_1"
hd_csv_path = hd_session_dir / "data/tc_comparison/hd_class3_ProbeB.csv"
prepare_hd_tc(
    hd_session_dir / "data/tuning_curves/ProbeB/tuning_curves.tc",
    hd_csv_path, bins=comparison_bins, hd_class=3, unit_prefix="m15:260630:B",
)
fm_m15_260630_B = load_tc(
    hd_csv_path,
    label="m15 260630 ProbeB HD", range=tcRange(False), response_units="Hz",
)
rf_session_dir = root / "m15/260630/260630_3"
rf_source = (rf_session_dir / "data/rfmapping/good/-100_400_1ms/ProbeB"
             / "regular_unitsSpikeCounts_260630_3.rfmap")
rf_csv_path = rf_session_dir / f"data/tc_comparison/rf_inhibitory_x_2d{rf_suffix}_ProbeB.csv"
prepare_rf_comparison(
    rf_source, rf_csv_path,
    projection_path=rf_session_dir / "data" / f"{rf_source.stem}_ProbeB{'_inhibitory' if rf_only else ''}_1d{rf_suffix}.csv",
    detected_rf_path=rf_result_path(rf_source, rf_type="inhibitory"),
    time_range_s=rf_response_window_s, rf_only=rf_only, unit_prefix="m15:260630:B",
)
rf_m15_260630_B = load_tc(
    rf_csv_path,
    response_units="spike_count", label="m15 260630 ProbeB inhibitory RF3",
)
rf_m15_260630_B = rf_pick(rf_m15_260630_B, max_zero_bins=rf_max_zero_bins)
rf_m15_260630_B = resample_profiles(rf_m15_260630_B, range=tcRange(True), bins=comparison_bins, fill_value=0)

fm_m15_260630 = concat(fm_m15_260630_A, fm_m15_260630_B, label="m15 260630 A+B HD")
rf_m15_260630 = concat(rf_m15_260630_A, rf_m15_260630_B, label="m15 260630 A+B inhibitory RF3")

units_m15_260630 = fm_m15_260630.index.intersection(rf_m15_260630.index)
hd_normalized_m15_260630 = normalize_tc(fm_m15_260630.loc[units_m15_260630])
rf_normalized_m15_260630 = normalize_tc(rf_m15_260630.loc[units_m15_260630])


In [ ]:
# Native angles, one explicitly selected reference per call.
sorted_fm_m15_260630 = prepare_comparison(hd_normalized_m15_260630, rf_normalized_m15_260630, mode="native", matched_min=True)
plot_comparison_heatmaps(sorted_fm_m15_260630, **plot_options)
sorted_rf_m15_260630 = prepare_comparison(rf_normalized_m15_260630, hd_normalized_m15_260630, mode="native", reference_min=True)
plot_comparison_heatmaps(sorted_rf_m15_260630, **plot_options)


In [ ]:
# Subtract the selected reference direction from both curves.
aligned_fm_m15_260630 = prepare_comparison(hd_normalized_m15_260630, rf_normalized_m15_260630, mode="aligned", matched_min=True, matched_fill_value=0)
plot_comparison_heatmaps(aligned_fm_m15_260630, **plot_options)
aligned_rf_m15_260630 = prepare_comparison(rf_normalized_m15_260630, hd_normalized_m15_260630, mode="aligned", reference_min=True, reference_fill_value=0)
plot_comparison_heatmaps(aligned_rf_m15_260630, **plot_options)


In [ ]:
# Center the reference; add its selected direction to the matched curve.
summed_fm_m15_260630 = prepare_comparison(hd_normalized_m15_260630, rf_normalized_m15_260630, mode="sum", is_wrap=is_wrap, matched_min=True, matched_fill_value=0)
plot_comparison_heatmaps(summed_fm_m15_260630, **plot_options)
summed_rf_m15_260630 = prepare_comparison(rf_normalized_m15_260630, hd_normalized_m15_260630, mode="sum", is_wrap=is_wrap, reference_min=True, reference_fill_value=0)
plot_comparison_heatmaps(summed_rf_m15_260630, **plot_options)


In [ ]:
peaks = summed_fm_m15_260630["peaks"]
peak_sum_deg = (peaks["reference_peak_deg"] + peaks["matched_peak_deg"]) % 360
peak_sum_rad = np.deg2rad(peak_sum_deg.dropna().to_numpy())

counts, edges = np.histogram(peak_sum_rad, bins=30, range=(0, 2 * np.pi))
centers_deg = np.rad2deg((edges[:-1] + edges[1:]) / 2)
# Equal occupancy defines a uniform null; retain all bins, including zeros.
rayleigh_score, rayleigh_p = rayleigh_test(counts, np.ones_like(counts), centers_deg)

with plt.rc_context(LIGHT_PLOT_STYLE):
    fig, ax = plt.subplots(figsize=(6, 6), subplot_kw={"projection": "polar"}, layout="constrained")
    ax.bar((edges[:-1] + edges[1:]) / 2, counts, width=np.diff(edges), edgecolor="white")
    ax.set_theta_zero_location("N")

    ax.set_theta_direction(-1)
    ax.yaxis.set_major_locator(MaxNLocator(integer=True))
    ax.set_xlabel("Unit count (radial axis)")
    ax.set_title(
        "m15 260630: HD peak + inhibitory RF minimum angles\n"
        f"n = {int(counts.sum())}\n"
        f"Rayleigh score = {rayleigh_score:.3f}, p = {rayleigh_p:.3g}",
        pad=20,
    )
    plt.show()


In [ ]:
peaks_m15_260630 = sorted_fm_m15_260630["peaks"].dropna(subset=["reference_peak_deg", "matched_peak_deg"])
statistics_m15_260630 = None
if len(peaks_m15_260630) >= 3:
    reference = peaks_m15_260630.reference_peak_deg.to_numpy()
    matched = peaks_m15_260630.matched_peak_deg.to_numpy()
    statistics_m15_260630 = compare_direction_angles(reference, matched, n_permutations=10_000, random_seed=1)
    statistics_figures_m15_260630 = plot_direction_comparison(
        reference, matched, statistics_m15_260630,
        reference_label=fm_m15_260630.attrs["label"], matched_label=rf_m15_260630.attrs["label"],
        reference_ticklabels=profile_coordinates(fm_m15_260630)["ticklabels"],
        matched_ticklabels=profile_coordinates(rf_m15_260630)["ticklabels"],
        show=False,
    )
    for figure, axis in statistics_figures_m15_260630:
        axis.set_title(fill(axis.get_title(), width=60))
        figure.tight_layout()
        display(figure)
        plt.close(figure)


## m15 260630 Probe A only: HD 1 / inhibitory RF 3

Use the same sessions and filters as A+B above, selecting Probe **A** only.


In [ ]:
units_m15_260630_A = fm_m15_260630_A.index.intersection(rf_m15_260630_A.index)
hd_normalized_m15_260630_A = normalize_tc(fm_m15_260630_A.loc[units_m15_260630_A])
rf_normalized_m15_260630_A = normalize_tc(rf_m15_260630_A.loc[units_m15_260630_A])

{"HD": len(fm_m15_260630_A), "RF": len(rf_m15_260630_A), "shared": len(units_m15_260630_A)}


In [ ]:
# Native angles, one explicitly selected reference per call.
sorted_fm_m15_260630_A = prepare_comparison(hd_normalized_m15_260630_A, rf_normalized_m15_260630_A, mode="native", matched_min=True)
plot_comparison_heatmaps(sorted_fm_m15_260630_A, **plot_options)
sorted_rf_m15_260630_A = prepare_comparison(rf_normalized_m15_260630_A, hd_normalized_m15_260630_A, mode="native", reference_min=True)
plot_comparison_heatmaps(sorted_rf_m15_260630_A, **plot_options)


In [ ]:
# Subtract the selected reference direction from both curves.
aligned_fm_m15_260630_A = prepare_comparison(hd_normalized_m15_260630_A, rf_normalized_m15_260630_A, mode="aligned", matched_min=True, matched_fill_value=0)
plot_comparison_heatmaps(aligned_fm_m15_260630_A, **plot_options)
aligned_rf_m15_260630_A = prepare_comparison(rf_normalized_m15_260630_A, hd_normalized_m15_260630_A, mode="aligned", reference_min=True, reference_fill_value=0)
plot_comparison_heatmaps(aligned_rf_m15_260630_A, **plot_options)


In [ ]:
# Center the reference; add its selected direction to the matched curve.
summed_fm_m15_260630_A = prepare_comparison(hd_normalized_m15_260630_A, rf_normalized_m15_260630_A, mode="sum", is_wrap=is_wrap, matched_min=True, matched_fill_value=0)
plot_comparison_heatmaps(summed_fm_m15_260630_A, **plot_options)
summed_rf_m15_260630_A = prepare_comparison(rf_normalized_m15_260630_A, hd_normalized_m15_260630_A, mode="sum", is_wrap=is_wrap, reference_min=True, reference_fill_value=0)
plot_comparison_heatmaps(summed_rf_m15_260630_A, **plot_options)


In [ ]:
peaks = summed_fm_m15_260630_A["peaks"]
peak_sum_deg = (peaks["reference_peak_deg"] + peaks["matched_peak_deg"]) % 360
peak_sum_rad = np.deg2rad(peak_sum_deg.dropna().to_numpy())

counts, edges = np.histogram(peak_sum_rad, bins=30, range=(0, 2 * np.pi))
centers_deg = np.rad2deg((edges[:-1] + edges[1:]) / 2)
# Equal occupancy defines a uniform null; retain all bins, including zeros.
rayleigh_score, rayleigh_p = rayleigh_test(counts, np.ones_like(counts), centers_deg)

with plt.rc_context(LIGHT_PLOT_STYLE):
    fig, ax = plt.subplots(figsize=(6, 6), subplot_kw={"projection": "polar"}, layout="constrained")
    ax.bar((edges[:-1] + edges[1:]) / 2, counts, width=np.diff(edges), edgecolor="white")
    ax.set_theta_zero_location("N")

    ax.set_theta_direction(-1)
    ax.yaxis.set_major_locator(MaxNLocator(integer=True))
    ax.set_xlabel("Unit count (radial axis)")
    ax.set_title(
        "m15 260630 Probe A: HD peak + inhibitory RF minimum angles\n"
        f"n = {int(counts.sum())}\n"
        f"Rayleigh score = {rayleigh_score:.3f}, p = {rayleigh_p:.3g}",
        pad=20,
    )
    plt.show()


In [ ]:
peaks_m15_260630_A = sorted_fm_m15_260630_A["peaks"].dropna(subset=["reference_peak_deg", "matched_peak_deg"])
statistics_m15_260630_A = None
if len(peaks_m15_260630_A) >= 3:
    reference = peaks_m15_260630_A.reference_peak_deg.to_numpy()
    matched = peaks_m15_260630_A.matched_peak_deg.to_numpy()
    statistics_m15_260630_A = compare_direction_angles(reference, matched, n_permutations=10_000, random_seed=1)
    statistics_figures_m15_260630_A = plot_direction_comparison(
        reference, matched, statistics_m15_260630_A,
        reference_label=fm_m15_260630_A.attrs["label"], matched_label=rf_m15_260630_A.attrs["label"],
        reference_ticklabels=profile_coordinates(fm_m15_260630_A)["ticklabels"],
        matched_ticklabels=profile_coordinates(rf_m15_260630_A)["ticklabels"],
        show=False,
    )
    for figure, axis in statistics_figures_m15_260630_A:
        axis.set_title(fill(axis.get_title(), width=60))
        figure.tight_layout()
        display(figure)
        plt.close(figure)


## m15 260630 Probe B only: HD 1 / inhibitory RF 3

Use the same sessions and filters as A+B above, selecting Probe **B** only.


In [ ]:
units_m15_260630_B = fm_m15_260630_B.index.intersection(rf_m15_260630_B.index)
hd_normalized_m15_260630_B = normalize_tc(fm_m15_260630_B.loc[units_m15_260630_B])
rf_normalized_m15_260630_B = normalize_tc(rf_m15_260630_B.loc[units_m15_260630_B])

{"HD": len(fm_m15_260630_B), "RF": len(rf_m15_260630_B), "shared": len(units_m15_260630_B)}


In [ ]:
# Native angles, one explicitly selected reference per call.
sorted_fm_m15_260630_B = prepare_comparison(hd_normalized_m15_260630_B, rf_normalized_m15_260630_B, mode="native", matched_min=True)
plot_comparison_heatmaps(sorted_fm_m15_260630_B, **plot_options)
sorted_rf_m15_260630_B = prepare_comparison(rf_normalized_m15_260630_B, hd_normalized_m15_260630_B, mode="native", reference_min=True)
plot_comparison_heatmaps(sorted_rf_m15_260630_B, **plot_options)


In [ ]:
# Subtract the selected reference direction from both curves.
aligned_fm_m15_260630_B = prepare_comparison(hd_normalized_m15_260630_B, rf_normalized_m15_260630_B, mode="aligned", matched_min=True, matched_fill_value=0)
plot_comparison_heatmaps(aligned_fm_m15_260630_B, **plot_options)
aligned_rf_m15_260630_B = prepare_comparison(rf_normalized_m15_260630_B, hd_normalized_m15_260630_B, mode="aligned", reference_min=True, reference_fill_value=0)
plot_comparison_heatmaps(aligned_rf_m15_260630_B, **plot_options)


In [ ]:
# Center the reference; add its selected direction to the matched curve.
summed_fm_m15_260630_B = prepare_comparison(hd_normalized_m15_260630_B, rf_normalized_m15_260630_B, mode="sum", is_wrap=is_wrap, matched_min=True, matched_fill_value=0)
plot_comparison_heatmaps(summed_fm_m15_260630_B, **plot_options)
summed_rf_m15_260630_B = prepare_comparison(rf_normalized_m15_260630_B, hd_normalized_m15_260630_B, mode="sum", is_wrap=is_wrap, reference_min=True, reference_fill_value=0)
plot_comparison_heatmaps(summed_rf_m15_260630_B, **plot_options)


In [ ]:
peaks = summed_fm_m15_260630_B["peaks"]
peak_sum_deg = (peaks["reference_peak_deg"] + peaks["matched_peak_deg"]) % 360
peak_sum_rad = np.deg2rad(peak_sum_deg.dropna().to_numpy())

counts, edges = np.histogram(peak_sum_rad, bins=30, range=(0, 2 * np.pi))
centers_deg = np.rad2deg((edges[:-1] + edges[1:]) / 2)
# Equal occupancy defines a uniform null; retain all bins, including zeros.
rayleigh_score, rayleigh_p = rayleigh_test(counts, np.ones_like(counts), centers_deg)

with plt.rc_context(LIGHT_PLOT_STYLE):
    fig, ax = plt.subplots(figsize=(6, 6), subplot_kw={"projection": "polar"}, layout="constrained")
    ax.bar((edges[:-1] + edges[1:]) / 2, counts, width=np.diff(edges), edgecolor="white")
    ax.set_theta_zero_location("N")

    ax.set_theta_direction(-1)
    ax.yaxis.set_major_locator(MaxNLocator(integer=True))
    ax.set_xlabel("Unit count (radial axis)")
    ax.set_title(
        "m15 260630 Probe B: HD peak + inhibitory RF minimum angles\n"
        f"n = {int(counts.sum())}\n"
        f"Rayleigh score = {rayleigh_score:.3f}, p = {rayleigh_p:.3g}",
        pad=20,
    )
    plt.show()


In [ ]:
peaks_m15_260630_B = sorted_fm_m15_260630_B["peaks"].dropna(subset=["reference_peak_deg", "matched_peak_deg"])
statistics_m15_260630_B = None
if len(peaks_m15_260630_B) >= 3:
    reference = peaks_m15_260630_B.reference_peak_deg.to_numpy()
    matched = peaks_m15_260630_B.matched_peak_deg.to_numpy()
    statistics_m15_260630_B = compare_direction_angles(reference, matched, n_permutations=10_000, random_seed=1)
    statistics_figures_m15_260630_B = plot_direction_comparison(
        reference, matched, statistics_m15_260630_B,
        reference_label=fm_m15_260630_B.attrs["label"], matched_label=rf_m15_260630_B.attrs["label"],
        reference_ticklabels=profile_coordinates(fm_m15_260630_B)["ticklabels"],
        matched_ticklabels=profile_coordinates(rf_m15_260630_B)["ticklabels"],
        show=False,
    )
    for figure, axis in statistics_figures_m15_260630_B:
        axis.set_title(fill(axis.get_title(), width=60))
        figure.tight_layout()
        display(figure)
        plt.close(figure)


## m19 260827: HD 9 / inhibitory RF 2

Probe **A**. HD class is selected by `hd_class`.


In [ ]:
hd_session_dir = root / "m19/260827/260827_9"
hd_csv_path = hd_session_dir / "data/tc_comparison/hd_class3_ProbeA.csv"
prepare_hd_tc(
    hd_session_dir / "data/tuning_curves/ProbeA/tuning_curves.tc",
    hd_csv_path, bins=comparison_bins, hd_class=3, unit_prefix="m19:260827:A",
)
fm_m19_260827 = load_tc(
    hd_csv_path,
    label="m19 260827 ProbeA HD", range=tcRange(False), response_units="Hz",
)
rf_session_dir = root / "m19/260827/260827_2"
rf_source = (rf_session_dir / "data/rfmapping/good/-100_400_1ms/ProbeA"
             / "regular_unitsSpikeCounts_260827_2.rfmap")
rf_csv_path = rf_session_dir / f"data/tc_comparison/rf_inhibitory_x_2d{rf_suffix}_ProbeA.csv"
prepare_rf_comparison(
    rf_source, rf_csv_path,
    projection_path=rf_session_dir / "data" / f"{rf_source.stem}_ProbeA{'_inhibitory' if rf_only else ''}_1d{rf_suffix}.csv",
    detected_rf_path=rf_result_path(rf_source, rf_type="inhibitory"),
    time_range_s=rf_response_window_s, rf_only=rf_only, unit_prefix="m19:260827:A",
)
rf_m19_260827 = load_tc(
    rf_csv_path,
    response_units="spike_count", label="m19 260827 ProbeA inhibitory RF2",
)
rf_m19_260827 = rf_pick(rf_m19_260827, max_zero_bins=rf_max_zero_bins)
rf_m19_260827 = resample_profiles(rf_m19_260827, range=tcRange(True), bins=comparison_bins, fill_value=0)

units_m19_260827 = fm_m19_260827.index.intersection(rf_m19_260827.index)
hd_normalized_m19_260827 = normalize_tc(fm_m19_260827.loc[units_m19_260827])
rf_normalized_m19_260827 = normalize_tc(rf_m19_260827.loc[units_m19_260827])


In [ ]:
if len(units_m19_260827):
    # Native angles, one explicitly selected reference per call.
    sorted_fm_m19_260827 = prepare_comparison(hd_normalized_m19_260827, rf_normalized_m19_260827, mode="native", matched_min=True)
    plot_comparison_heatmaps(sorted_fm_m19_260827, **plot_options)
    sorted_rf_m19_260827 = prepare_comparison(rf_normalized_m19_260827, hd_normalized_m19_260827, mode="native", reference_min=True)
    plot_comparison_heatmaps(sorted_rf_m19_260827, **plot_options)


In [ ]:
if len(units_m19_260827):
    # Subtract the selected reference direction from both curves.
    aligned_fm_m19_260827 = prepare_comparison(hd_normalized_m19_260827, rf_normalized_m19_260827, mode="aligned", matched_min=True, matched_fill_value=0)
    plot_comparison_heatmaps(aligned_fm_m19_260827, **plot_options)
    aligned_rf_m19_260827 = prepare_comparison(rf_normalized_m19_260827, hd_normalized_m19_260827, mode="aligned", reference_min=True, reference_fill_value=0)
    plot_comparison_heatmaps(aligned_rf_m19_260827, **plot_options)


In [ ]:
if len(units_m19_260827):
    # Center the reference; add its selected direction to the matched curve.
    summed_fm_m19_260827 = prepare_comparison(hd_normalized_m19_260827, rf_normalized_m19_260827, mode="sum", is_wrap=is_wrap, matched_min=True, matched_fill_value=0)
    plot_comparison_heatmaps(summed_fm_m19_260827, **plot_options)
    summed_rf_m19_260827 = prepare_comparison(rf_normalized_m19_260827, hd_normalized_m19_260827, mode="sum", is_wrap=is_wrap, reference_min=True, reference_fill_value=0)
    plot_comparison_heatmaps(summed_rf_m19_260827, **plot_options)


In [ ]:
if len(units_m19_260827):
    peaks = summed_fm_m19_260827["peaks"]
    peak_sum_deg = (peaks["reference_peak_deg"] + peaks["matched_peak_deg"]) % 360
    peak_sum_rad = np.deg2rad(peak_sum_deg.dropna().to_numpy())

    counts, edges = np.histogram(peak_sum_rad, bins=30, range=(0, 2 * np.pi))
    centers_deg = np.rad2deg((edges[:-1] + edges[1:]) / 2)
    # Equal occupancy defines a uniform null; retain all bins, including zeros.
    rayleigh_score, rayleigh_p = rayleigh_test(counts, np.ones_like(counts), centers_deg)

    with plt.rc_context(LIGHT_PLOT_STYLE):
        fig, ax = plt.subplots(figsize=(6, 6), subplot_kw={"projection": "polar"}, layout="constrained")
        ax.bar((edges[:-1] + edges[1:]) / 2, counts, width=np.diff(edges), edgecolor="white")
        ax.set_theta_zero_location("N")

        ax.set_theta_direction(-1)
        ax.yaxis.set_major_locator(MaxNLocator(integer=True))
        ax.set_xlabel("Unit count (radial axis)")
        ax.set_title(
            "m19 260827: HD peak + inhibitory RF minimum angles\n"
            f"n = {int(counts.sum())}\n"
            f"Rayleigh score = {rayleigh_score:.3f}, p = {rayleigh_p:.3g}",
            pad=20,
        )
        plt.show()


In [ ]:
if len(units_m19_260827):
    peaks_m19_260827 = sorted_fm_m19_260827["peaks"].dropna(subset=["reference_peak_deg", "matched_peak_deg"])
    statistics_m19_260827 = None
    if len(peaks_m19_260827) >= 3:
        reference = peaks_m19_260827.reference_peak_deg.to_numpy()
        matched = peaks_m19_260827.matched_peak_deg.to_numpy()
        statistics_m19_260827 = compare_direction_angles(reference, matched, n_permutations=10_000, random_seed=1)
        statistics_figures_m19_260827 = plot_direction_comparison(
            reference, matched, statistics_m19_260827,
            reference_label=fm_m19_260827.attrs["label"], matched_label=rf_m19_260827.attrs["label"],
            reference_ticklabels=profile_coordinates(fm_m19_260827)["ticklabels"],
            matched_ticklabels=profile_coordinates(rf_m19_260827)["ticklabels"],
            show=False,
        )
        for figure, axis in statistics_figures_m19_260827:
            axis.set_title(fill(axis.get_title(), width=60))
            figure.tight_layout()
            display(figure)
            plt.close(figure)


## m20 260921: HD 9 / inhibitory RF 2

Probe **A**. HD class is selected by `hd_class`.


In [ ]:
hd_session_dir = root / "m20/260921/260921_9"
hd_csv_path = hd_session_dir / "data/tc_comparison/hd_class3_ProbeA.csv"
prepare_hd_tc(
    hd_session_dir / "data/tuning_curves/ProbeA/tuning_curves.tc",
    hd_csv_path, bins=comparison_bins, hd_class=3, unit_prefix="m20:260921:A",
)
fm_m20_260921 = load_tc(
    hd_csv_path,
    label="m20 260921 ProbeA HD", range=tcRange(False), response_units="Hz",
)
rf_session_dir = root / "m20/260921/260921_2"
rf_source = (rf_session_dir / "data/rfmapping/good/-100_400_1ms/ProbeA"
             / "regular_unitsSpikeCounts_260921_2.rfmap")
rf_csv_path = rf_session_dir / f"data/tc_comparison/rf_inhibitory_x_2d{rf_suffix}_ProbeA.csv"
prepare_rf_comparison(
    rf_source, rf_csv_path,
    projection_path=rf_session_dir / "data" / f"{rf_source.stem}_ProbeA{'_inhibitory' if rf_only else ''}_1d{rf_suffix}.csv",
    detected_rf_path=rf_result_path(rf_source, rf_type="inhibitory"),
    time_range_s=rf_response_window_s, rf_only=rf_only, unit_prefix="m20:260921:A",
)
rf_m20_260921 = load_tc(
    rf_csv_path,
    response_units="spike_count", label="m20 260921 ProbeA inhibitory RF2",
)
rf_m20_260921 = rf_pick(rf_m20_260921, max_zero_bins=rf_max_zero_bins)
rf_m20_260921 = resample_profiles(rf_m20_260921, range=tcRange(True), bins=comparison_bins, fill_value=0)

units_m20_260921 = fm_m20_260921.index.intersection(rf_m20_260921.index)
hd_normalized_m20_260921 = normalize_tc(fm_m20_260921.loc[units_m20_260921])
rf_normalized_m20_260921 = normalize_tc(rf_m20_260921.loc[units_m20_260921])


In [ ]:
# Native angles, one explicitly selected reference per call.
sorted_fm_m20_260921 = prepare_comparison(hd_normalized_m20_260921, rf_normalized_m20_260921, mode="native", matched_min=True)
plot_comparison_heatmaps(sorted_fm_m20_260921, **plot_options)
sorted_rf_m20_260921 = prepare_comparison(rf_normalized_m20_260921, hd_normalized_m20_260921, mode="native", reference_min=True)
plot_comparison_heatmaps(sorted_rf_m20_260921, **plot_options)


In [ ]:
# Subtract the selected reference direction from both curves.
aligned_fm_m20_260921 = prepare_comparison(hd_normalized_m20_260921, rf_normalized_m20_260921, mode="aligned", matched_min=True, matched_fill_value=0)
plot_comparison_heatmaps(aligned_fm_m20_260921, **plot_options)
aligned_rf_m20_260921 = prepare_comparison(rf_normalized_m20_260921, hd_normalized_m20_260921, mode="aligned", reference_min=True, reference_fill_value=0)
plot_comparison_heatmaps(aligned_rf_m20_260921, **plot_options)


In [ ]:
# Center the reference; add its selected direction to the matched curve.
summed_fm_m20_260921 = prepare_comparison(hd_normalized_m20_260921, rf_normalized_m20_260921, mode="sum", is_wrap=is_wrap, matched_min=True, matched_fill_value=0)
plot_comparison_heatmaps(summed_fm_m20_260921, **plot_options)
summed_rf_m20_260921 = prepare_comparison(rf_normalized_m20_260921, hd_normalized_m20_260921, mode="sum", is_wrap=is_wrap, reference_min=True, reference_fill_value=0)
plot_comparison_heatmaps(summed_rf_m20_260921, **plot_options)


In [ ]:
peaks = summed_fm_m20_260921["peaks"]
peak_sum_deg = (peaks["reference_peak_deg"] + peaks["matched_peak_deg"]) % 360
peak_sum_rad = np.deg2rad(peak_sum_deg.dropna().to_numpy())

counts, edges = np.histogram(peak_sum_rad, bins=30, range=(0, 2 * np.pi))
centers_deg = np.rad2deg((edges[:-1] + edges[1:]) / 2)
# Equal occupancy defines a uniform null; retain all bins, including zeros.
rayleigh_score, rayleigh_p = rayleigh_test(counts, np.ones_like(counts), centers_deg)

with plt.rc_context(LIGHT_PLOT_STYLE):
    fig, ax = plt.subplots(figsize=(6, 6), subplot_kw={"projection": "polar"}, layout="constrained")
    ax.bar((edges[:-1] + edges[1:]) / 2, counts, width=np.diff(edges), edgecolor="white")
    ax.set_theta_zero_location("N")

    ax.set_theta_direction(-1)
    ax.yaxis.set_major_locator(MaxNLocator(integer=True))
    ax.set_xlabel("Unit count (radial axis)")
    ax.set_title(
        "m20 260921: HD peak + inhibitory RF minimum angles\n"
        f"n = {int(counts.sum())}\n"
        f"Rayleigh score = {rayleigh_score:.3f}, p = {rayleigh_p:.3g}",
        pad=20,
    )
    plt.show()


In [ ]:
peaks_m20_260921 = sorted_fm_m20_260921["peaks"].dropna(subset=["reference_peak_deg", "matched_peak_deg"])
statistics_m20_260921 = None
if len(peaks_m20_260921) >= 3:
    reference = peaks_m20_260921.reference_peak_deg.to_numpy()
    matched = peaks_m20_260921.matched_peak_deg.to_numpy()
    statistics_m20_260921 = compare_direction_angles(reference, matched, n_permutations=10_000, random_seed=1)
    statistics_figures_m20_260921 = plot_direction_comparison(
        reference, matched, statistics_m20_260921,
        reference_label=fm_m20_260921.attrs["label"], matched_label=rf_m20_260921.attrs["label"],
        reference_ticklabels=profile_coordinates(fm_m20_260921)["ticklabels"],
        matched_ticklabels=profile_coordinates(rf_m20_260921)["ticklabels"],
        show=False,
    )
    for figure, axis in statistics_figures_m20_260921:
        axis.set_title(fill(axis.get_title(), width=60))
        figure.tight_layout()
        display(figure)
        plt.close(figure)


## All four mice: m14 + m15 + m19 + m20

Pool the mouse-level tables loaded with `rf_type` above, including both m15 probes, then apply the
same HD3 and RF zero-bin limit and select their shared units explicitly.
Rows are stacked without averaging; `(mouse, date, probe, unit_id)` remains unique.


In [ ]:
fm_all = concat(fm_m14_260609, fm_m15_260630, fm_m19_260827, fm_m20_260921, label="All mice HD")
# Use the common grid for pooling; retain the existing zero padding outside m14 coverage.
rf_m14_common = resample_profiles(rf_m14_260609, range=tcRange(True), bins=comparison_bins, fill_value=0)
rf_all = concat(rf_m14_common, rf_m15_260630, rf_m19_260827, rf_m20_260921, label="All mice inhibitory RF")

units_all = fm_all.index.intersection(rf_all.index)
hd_normalized_all = normalize_tc(fm_all.loc[units_all])
rf_normalized_all = normalize_tc(rf_all.loc[units_all])


In [ ]:
# Native angles, one explicitly selected reference per call.
sorted_fm_all = prepare_comparison(hd_normalized_all, rf_normalized_all, mode="native", matched_min=True)
plot_comparison_heatmaps(sorted_fm_all, **plot_options)
sorted_rf_all = prepare_comparison(rf_normalized_all, hd_normalized_all, mode="native", reference_min=True)
plot_comparison_heatmaps(sorted_rf_all, **plot_options)


In [ ]:
# Subtract the selected reference direction from both curves.
aligned_fm_all = prepare_comparison(hd_normalized_all, rf_normalized_all, mode="aligned", matched_min=True, matched_fill_value=0)
plot_comparison_heatmaps(aligned_fm_all, **plot_options)
aligned_rf_all = prepare_comparison(rf_normalized_all, hd_normalized_all, mode="aligned", reference_min=True, reference_fill_value=0)
plot_comparison_heatmaps(aligned_rf_all, **plot_options)


In [ ]:
# Center the reference; add its selected direction to the matched curve.
summed_fm_all = prepare_comparison(hd_normalized_all, rf_normalized_all, mode="sum", is_wrap=is_wrap, matched_min=True, matched_fill_value=0)
plot_comparison_heatmaps(summed_fm_all, **plot_options)
summed_rf_all = prepare_comparison(rf_normalized_all, hd_normalized_all, mode="sum", is_wrap=is_wrap, reference_min=True, reference_fill_value=0)
plot_comparison_heatmaps(summed_rf_all, **plot_options)


In [ ]:
peaks = summed_fm_all["peaks"]
peak_sum_deg = (peaks["reference_peak_deg"] + peaks["matched_peak_deg"]) % 360
peak_sum_rad = np.deg2rad(peak_sum_deg.dropna().to_numpy())

counts, edges = np.histogram(peak_sum_rad, bins=30, range=(0, 2 * np.pi))
centers_deg = np.rad2deg((edges[:-1] + edges[1:]) / 2)
# Equal occupancy defines a uniform null; retain all bins, including zeros.
rayleigh_score, rayleigh_p = rayleigh_test(counts, np.ones_like(counts), centers_deg)

with plt.rc_context(LIGHT_PLOT_STYLE):
    fig, ax = plt.subplots(figsize=(6, 6), subplot_kw={"projection": "polar"}, layout="constrained")
    ax.bar((edges[:-1] + edges[1:]) / 2, counts, width=np.diff(edges), edgecolor="white")
    ax.set_theta_zero_location("N")

    ax.set_theta_direction(-1)
    ax.yaxis.set_major_locator(MaxNLocator(integer=True))
    ax.set_xlabel("Unit count (radial axis)")
    ax.set_title(
        "All mice: HD peak + inhibitory RF minimum angles\n"
        f"n = {int(counts.sum())}\n"
        f"Rayleigh score = {rayleigh_score:.3f}, p = {rayleigh_p:.3g}",
        pad=20,
    )
    plt.show()


In [ ]:
peaks_all = sorted_fm_all["peaks"].dropna(subset=["reference_peak_deg", "matched_peak_deg"])
statistics_all = None
if len(peaks_all) >= 3:
    reference = peaks_all.reference_peak_deg.to_numpy()
    matched = peaks_all.matched_peak_deg.to_numpy()
    statistics_all = compare_direction_angles(reference, matched, n_permutations=10_000, random_seed=1)
    statistics_figures_all = plot_direction_comparison(
        reference, matched, statistics_all,
        reference_label=fm_all.attrs["label"], matched_label=rf_all.attrs["label"],
        reference_ticklabels=profile_coordinates(fm_all)["ticklabels"],
        matched_ticklabels=profile_coordinates(rf_all)["ticklabels"],
        show=False,
    )
    for figure, axis in statistics_figures_all:
        axis.set_title(fill(axis.get_title(), width=60))
        figure.tight_layout()
        display(figure)
        plt.close(figure)
